# OnePLM Data Exploration Notebook

This notebook helps you explore Windchill PLM data stored in the local SQLite database
and prototype validation rules before formalizing them into `config/checks.json`.

**Prerequisites:**
```bash
pip install -e ".[notebook]"
oneplm init
oneplm sync
```

In [ ]:
from oneplm_ingestion.dataframe import load_objects, load_check_results, load_sync_log, load_pdfs

# Change this if your database is in a different location
DB_PATH = "../data/oneplm.db"

## 1. Database Status
Check what has been synced and when.

In [ ]:
sync_log = load_sync_log(DB_PATH)
sync_log

## 2. Explore Objects by Type
Load all objects and see what types are present.

In [ ]:
df = load_objects(DB_PATH)
print(f"Total objects: {len(df)}")
df.groupby("type_name").size().reset_index(name="count")

In [ ]:
# Load just one type -- change the type name to explore different types
parts = load_objects(DB_PATH, type_name="Part PDP")
parts.head()

In [ ]:
# See all columns available (base columns + expanded attributes)
print(f"Columns ({len(parts.columns)}):")
for col in sorted(parts.columns):
    print(f"  {col}")

## 3. Find Missing Values
Identify which attributes have missing or empty values -- useful for building `not_empty` rules.

In [ ]:
# Count nulls/empty strings per column
missing = parts.isnull().sum() + (parts == "").sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

## 4. Prototype Check Logic

Before writing a JSON rule, test the logic here interactively.

### Example: "When State is Released, ApprovalDate must not be empty"

In [ ]:
# Filter to Released parts (adjust column name if needed)
state_col = "State" if "State" in parts.columns else "state"
released = parts[parts[state_col] == "Released"]
print(f"Released parts: {len(released)}")

# Check which ones have empty/missing ApprovalDate
if "ApprovalDate" in released.columns:
    missing_approval = released[
        released["ApprovalDate"].isnull() | (released["ApprovalDate"] == "")
    ]
    print(f"Released parts missing ApprovalDate: {len(missing_approval)}")
    missing_approval[["number", "name", state_col, "ApprovalDate"]]
else:
    print("ApprovalDate column not found -- check column names above")

### Example: Compare Part PDP and IFU Document attributes
Pair objects by Number and compare a shared attribute.

In [ ]:
parts = load_objects(DB_PATH, type_name="Part PDP")
ifus = load_objects(DB_PATH, type_name="IFU Document")

# Merge on number to pair them
merged = parts.merge(ifus, on="number", suffixes=("_part", "_ifu"), how="inner")
print(f"Paired objects: {len(merged)}")

# Compare an attribute across types (adjust column name as needed)
if "RegulatoryClass_part" in merged.columns and "RegulatoryClass_ifu" in merged.columns:
    mismatches = merged[
        merged["RegulatoryClass_part"] != merged["RegulatoryClass_ifu"]
    ]
    print(f"Mismatches: {len(mismatches)}")
    mismatches[["number", "RegulatoryClass_part", "RegulatoryClass_ifu"]]

## 5. Review Check Results
After running `oneplm check`, inspect results here.

In [ ]:
results = load_check_results(DB_PATH)
if len(results) > 0:
    print(f"Total results: {len(results)}")
    display(results.groupby(["check_name", "passed"]).size().unstack(fill_value=0))
else:
    print("No check results yet. Run: oneplm check")

In [ ]:
# Show failed checks only
failed = load_check_results(DB_PATH, failed_only=True)
if len(failed) > 0:
    failed[["check_name", "source_object_id", "source_attr", "source_value", "target_value", "message"]]
else:
    print("No failures found.")

## 6. Formalizing Rules

Once you've identified a pattern above, translate it into a JSON rule in `config/checks.json`.

**Template:**
```json
{
  "name": "your_rule_name",
  "description": "Human-readable description",
  "source_type": "Part PDP",
  "target_type": "Part PDP",
  "match_on": "Number",
  "comparisons": [
    {
      "source_attr": "YourAttribute",
      "operator": "not_empty",
      "when": {
        "attr": "State.Value",
        "operator": "equals",
        "value": "Released"
      }
    }
  ]
}
```

**Available operators:** `equals`, `not_equals`, `contains`, `not_contains`,
`not_empty`, `is_empty`, `matches` (regex), `greater_than`, `less_than`,
`greater_equal`, `less_equal`, `before` (date), `after` (date)

Then run:
```bash
oneplm check
oneplm export checks -o check_results.csv
```